In [ ]:
import numpy as np
import pandas as pd
import math


# ============================================================
# Parameters
# ============================================================

lambda_add = 1.2
lambda_dec = 1.5

rng = np.random.default_rng(42)


# ============================================================
# Fast simulation: only returns who hits zero first
# ============================================================

def simulate_first_hit_side(
    q_ask0: int,
    q_bid0: int,
    lambda_add: float,
    lambda_dec: float,
    rng: np.random.Generator,
    max_events: int = 5_000_000,
) -> str:
    """
    Simulates two independent queues until one hits zero.

    Returns:
        "ask" if tau_ask < tau_bid
        "bid" if tau_bid < tau_ask
    """

    qa = int(q_ask0)
    qb = int(q_bid0)

    if qa <= 0:
        return "ask"
    if qb <= 0:
        return "bid"

    rate = lambda_add + lambda_dec
    p_up = lambda_add / rate

    for _ in range(max_events):
        # Choose which queue jumps
        if rng.random() < 0.5:
            # Ask jumps
            qa += 1 if rng.random() < p_up else -1
            if qa <= 0:
                return "ask"
        else:
            # Bid jumps
            qb += 1 if rng.random() < p_up else -1
            if qb <= 0:
                return "bid"

    raise RuntimeError("max_events reached before either queue hit zero")


# ============================================================
# Naive Monte Carlo estimator
# ============================================================

def estimate_prob_ask_before_bid_naive(
    q_ask0: int,
    q_bid0: int,
    lambda_add: float,
    lambda_dec: float,
    n_samples: int,
    rng: np.random.Generator,
) -> dict:
    """
    Estimates P(tau_ask < tau_bid) by naive Monte Carlo.
    """

    hits = 0

    for _ in range(n_samples):
        first = simulate_first_hit_side(
            q_ask0=q_ask0,
            q_bid0=q_bid0,
            lambda_add=lambda_add,
            lambda_dec=lambda_dec,
            rng=rng,
        )
        hits += (first == "ask")

    p_hat = hits / n_samples

    # Standard error for Bernoulli MC
    se = math.sqrt(max(p_hat * (1.0 - p_hat), 0.0) / n_samples)

    return {
        "q_ask0": q_ask0,
        "q_bid0": q_bid0,
        "n_samples": n_samples,
        "hits": hits,
        "p_hat": p_hat,
        "se": se,
        "ci95_low": max(0.0, p_hat - 1.96 * se),
        "ci95_high": min(1.0, p_hat + 1.96 * se),
    }


# ============================================================
# Grid search for rare-event region
# ============================================================

def grid_search_rare_region(
    ask_values,
    bid_values,
    lambda_add: float,
    lambda_dec: float,
    n_samples: int = 50_000,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Runs naive Monte Carlo on a grid of initial conditions.

    Goal:
        Find values where P(tau_ask < tau_bid) is around 1e-4.
    """

    rng = np.random.default_rng(seed)
    rows = []

    total = len(ask_values) * len(bid_values)
    k = 0

    for q_ask0 in ask_values:
        for q_bid0 in bid_values:
            k += 1
            print(f"[{k}/{total}] q_ask0={q_ask0}, q_bid0={q_bid0}")

            res = estimate_prob_ask_before_bid_naive(
                q_ask0=q_ask0,
                q_bid0=q_bid0,
                lambda_add=lambda_add,
                lambda_dec=lambda_dec,
                n_samples=n_samples,
                rng=rng,
            )

            rows.append(res)

    df = pd.DataFrame(rows)
    df = df.sort_values("p_hat").reset_index(drop=True)

    return df


# ============================================================
# Example: large ask size, small bid size
# ============================================================

ask_values = np.arange(80, 101, 10)   # large initial ask sizes
bid_values = np.arange(2, 6, 2)     # small initial bid sizes

df = grid_search_rare_region(
    ask_values=ask_values,
    bid_values=bid_values,
    lambda_add=lambda_add,
    lambda_dec=lambda_dec,
    n_samples=50_000,
    seed=123,
)

print(df)


# ============================================================
# Filter points close to probability 1e-4
# ============================================================

target = 1e-4

df_near = df[
    (df["p_hat"] >= target / 5)
    & (df["p_hat"] <= target * 5)
].copy()

df_near = df_near.sort_values("p_hat")

print("\nPoints with estimated probability around 1e-4:")
print(df_near)

[1/6] q_ask0=80, q_bid0=2
[2/6] q_ask0=80, q_bid0=4
[3/6] q_ask0=90, q_bid0=2
[4/6] q_ask0=90, q_bid0=4
[5/6] q_ask0=100, q_bid0=2
[6/6] q_ask0=100, q_bid0=4
   q_ask0  q_bid0  n_samples  hits    p_hat        se  ci95_low  ci95_high
0     100       2      50000     1  0.00002  0.000020  0.000000   0.000059
1     100       4      50000     6  0.00012  0.000049  0.000024   0.000216
2      90       2      50000     9  0.00018  0.000060  0.000062   0.000298
3      80       2      50000    13  0.00026  0.000072  0.000119   0.000401
4      90       4      50000    13  0.00026  0.000072  0.000119   0.000401
5      80       4      50000    44  0.00088  0.000133  0.000620   0.001140

Points with estimated probability around 1e-4:
   q_ask0  q_bid0  n_samples  hits    p_hat        se  ci95_low  ci95_high
0     100       2      50000     1  0.00002  0.000020  0.000000   0.000059
1     100       4      50000     6  0.00012  0.000049  0.000024   0.000216
2      90       2      50000     9  0.00018 